# MR3a — Daily Final Join (Reduce-Side)
**Inputs**: `mr3/daily_crash_mr.csv` + `mr3/address_store_mr.csv` + `mr3/type_count_store_mr.csv`
**Output**: `mr3/mr_daily_final.csv`

Join type: **inner** on `zone_id` (reduce-side join)
Mapper tags each record by source. Shuffle groups by zone_id. Reducer merges crash + store records.
Crash-only zones (no matching store) are dropped — remain in `daily_crash_mr.csv` for the map's blue layer.

In [1]:
import json
import pandas as pd
from collections import defaultdict

CRASH_INPUT  = 'daily_crash_mr.csv'
ADDR_INPUT   = 'address_store_mr.csv'
TYPE_INPUT   = 'type_count_store_mr.csv'
OUTPUT       = 'mr_daily_final.csv'

crashes = pd.read_csv(CRASH_INPUT)
addr    = pd.read_csv(ADDR_INPUT)
types   = pd.read_csv(TYPE_INPUT)

print(f'Crash rows : {len(crashes):,}  |  zones: {crashes["zone_id"].nunique():,}')
print(f'Store zones: {len(addr):,}')
print(f'Type rows  : {len(types):,}')

Crash rows : 2,891  |  zones: 1,490
Store zones: 1,823
Type rows  : 6,576


In [2]:
# ── MAP ───────────────────────────────────────────────────────────────────────
# Tag each record with its source, emit (zone_id, tagged_record)

# Collapse type rows into one dict per zone before mapping
type_by_zone = defaultdict(dict)
for _, row in types.iterrows():
    type_by_zone[str(row['zone_id'])][row['method_of_operation']] = int(row['count'])

def mapper_crash(row):
    return str(row['zone_id']), {
        'src':        'crash',
        'crash_date': row['crash_date'],
        'crashes':    row['crashes'],
        'injured':    row['injured'],
        'killed':     row['killed'],
    }

def mapper_store(row):
    zid = str(row['zone_id'])
    return zid, {
        'src':               'store',
        'store_count':       row['store_count'],
        'active_licenses':   row['active_licenses'],
        'outdated_licenses': row['outdated_licenses'],
        'type_counts_json':  json.dumps(type_by_zone.get(zid, {})),
    }

mapped  = [mapper_crash(row) for _, row in crashes.iterrows()]
mapped += [mapper_store(row) for _, row in addr.iterrows()]

print(f'Total mapped pairs: {len(mapped):,}')

Total mapped pairs: 4,714


In [3]:
# ── SHUFFLE ───────────────────────────────────────────────────────────────────
# Group all tagged records by zone_id

shuffled = defaultdict(list)
for zone_id, record in mapped:
    shuffled[zone_id].append(record)

print(f'Unique zone_ids after shuffle: {len(shuffled):,}')

Unique zone_ids after shuffle: 2,289


In [4]:
# ── REDUCE ────────────────────────────────────────────────────────────────────
# For each zone_id: find store record, join with all crash records.
# Zones with no matching store are dropped (crash-only zones).

def reducer(zone_id, records):
    store_rec  = next((r for r in records if r['src'] == 'store'), None)
    crash_recs = [r for r in records if r['src'] == 'crash']
    if not store_rec or not crash_recs:
        return []
    return [
        {
            'zone_id':           zone_id,
            'crash_date':        c['crash_date'],
            'crashes':           c['crashes'],
            'injured':           c['injured'],
            'killed':            c['killed'],
            'store_count':       store_rec['store_count'],
            'active_licenses':   store_rec['active_licenses'],
            'outdated_licenses': store_rec['outdated_licenses'],
            'type_counts_json':  store_rec['type_counts_json'],
        }
        for c in crash_recs
    ]

rows = []
for zone_id, records in shuffled.items():
    rows.extend(reducer(zone_id, records))

out = pd.DataFrame(rows).sort_values(['zone_id','crash_date']).reset_index(drop=True)
out.to_csv(OUTPUT, index=False)

print(f'Output rows : {len(out):,}')
print(f'Unique zones: {out["zone_id"].nunique():,}')
print(f'Dropped crash-only zones: {crashes["zone_id"].nunique() - out["zone_id"].nunique():,}')
print(f'Saved → {OUTPUT}')
out.head()

Output rows : 2,107
Unique zones: 1,024
Dropped crash-only zones: 466
Saved → mr_daily_final.csv


,zone_id,crash_date,crashes,injured,killed,store_count,active_licenses,outdated_licenses,type_counts_json
0,8a2a1008804ffff,2015-11-14,1,0.0,0.0,3,0,3,"{""Tavern Serving Beer Cider Wine And Liquor"": ..."
1,8a2a1008804ffff,2018-12-04,1,1.0,0.0,3,0,3,"{""Tavern Serving Beer Cider Wine And Liquor"": ..."
2,8a2a1008814ffff,2013-06-22,1,0.0,0.0,6,1,5,"{""Restaurant Serving Beer, Cider And Wine"": 1,..."
3,8a2a1008814ffff,2017-03-11,1,0.0,0.0,6,1,5,"{""Restaurant Serving Beer, Cider And Wine"": 1,..."
4,8a2a1008814ffff,2017-05-12,1,1.0,0.0,6,1,5,"{""Restaurant Serving Beer, Cider And Wine"": 1,..."
